# Task 1 — Exploratory Data Analysis
**ACIS Insurance Risk Analytics**

Objectives:
1. Summarise the dataset and assess data quality.
2. Compute the portfolio **Loss Ratio** and slice by Province / VehicleType / Gender.
3. Surface distributions, outliers, temporal trends, and geographic patterns.
4. Produce ≥ 3 insight-driven visualisations.


In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_insurance_data, add_derived_metrics
import src.eda_utils as eu

sns.set_theme(style="whitegrid", palette="muted")
FIGURES = pathlib.Path.cwd().parent / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

DATA_PATH = '../data/insurance_data_synth_cleaned.csv'
df = load_insurance_data(DATA_PATH)
df = add_derived_metrics(df)
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
df.head(3)


Loaded 20,000 rows × 55 columns


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims,LossRatio,Margin,HasClaim
0,1,397902,2015-06-01,True,South Africa,Individual,Mrs,English,Capitec,Current,...,Group A,Motor,Mobility Metered,Commercial,IFRS Constant,1796.31,0.00,0.000000,1796.31,0
1,2,766589,2015-04-01,False,South Africa,Pty Ltd,Mr,English,FNB,Savings,...,Group C,Motor,Standard Auto,Commercial,IFRS Constant,893.90,20922.47,23.405828,-20028.57,1
2,3,119424,2014-08-01,False,South Africa,Individual,Mr,English,Capitec,Current,...,Group C,Motor,Standard Auto,Commercial,IFRS Constant,2118.13,0.00,0.000000,2118.13,0


## 1. Data Summarisation

In [2]:
print("=== Shape ===")
print(df.shape)
print("\n=== dtypes (value counts) ===")
print(df.dtypes.value_counts())
print("\n=== Date range ===")
if 'TransactionMonth' in df.columns:
    print(df['TransactionMonth'].agg(['min', 'max']))


=== Shape ===
(20000, 55)

=== dtypes (value counts) ===
float64           10
int64              8
category           5
category           3
datetime64[us]     2
category           2
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
category           1
Name: count, dtype: int64

=== Date range ===
min   2014-02-01
max   2015-08-01
Name: TransactionMonth, dtype: datetime64[us]


In [3]:
eu.numeric_summary(df)


,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
UnderwrittenCoverID,20000.0,10000.500000,5773.647028,1.00000,5000.7500,10000.500,15000.250,2.000000e+04,0.000000,-1.200000
PolicyID,20000.0,548236.817900,259610.619985,100002.00000,324193.5000,546853.000,771965.250,9.999920e+05,0.007409,-1.197091
Mmcode,20000.0,55097.136100,25905.976638,10014.00000,32932.2500,55011.000,77577.750,9.999700e+04,-0.005535,-1.191113
RegistrationYear,20000.0,2006.041500,4.916725,1998.00000,2002.0000,2006.000,2010.000,2.014000e+03,-0.018538,-1.215080
Cylinders,20000.0,4.556850,1.167149,3.00000,4.0000,4.000,6.000,8.000000e+00,1.493188,1.523874
Cubiccapacity,20000.0,1903.419550,1221.777554,20.00000,1025.7500,1628.000,2483.000,1.233800e+04,1.418838,3.024075
Kilowatts,20000.0,90.349500,52.024756,1.30000,52.1000,80.500,118.100,4.896000e+02,1.166427,2.129414
NumberOfDoors,20000.0,4.197300,0.870178,2.00000,4.0000,4.000,5.000,5.000000e+00,-1.301364,1.347248
CustomValueEstimate,19200.0,256270.971615,151727.712623,3929.00000,145062.0000,226659.000,334893.250,1.356210e+06,1.249224,2.446545
CapitalOutstanding,20000.0,94981.786600,84053.218537,2.00000,33327.5000,72593.000,132684.750,7.275270e+05,1.693818,4.294383


## 2. Data Quality Assessment

In [4]:
miss = eu.missing_value_report(df)
print(f"Columns with missing values: {len(miss)}")
miss


Columns with missing values: 3


,missing,pct,dtype
CustomValueEstimate,800,4.0,float64
Bank,400,2.0,category
Bodytype,200,1.0,category


## 3. Portfolio Loss Ratio

In [5]:
portfolio_lr = df['TotalClaims'].sum() / df['TotalPremium'].sum()
total_margin = df['Margin'].sum()
claim_freq   = df['HasClaim'].mean()
claimants    = df[df['HasClaim'] == 1]
claim_sev    = claimants['TotalClaims'].mean()

print(f"Portfolio Loss Ratio  : {portfolio_lr:.4f}")
print(f"Total Margin (ZAR)    : R {total_margin:,.0f}")
print(f"Claim Frequency       : {claim_freq:.2%}")
print(f"Mean Claim Severity   : R {claim_sev:,.0f}")


Portfolio Loss Ratio  : 1.0760
Total Margin (ZAR)    : R -3,042,180
Claim Frequency       : 15.80%
Mean Claim Severity   : R 13,632


In [6]:
for col in ['Province', 'VehicleType', 'Gender']:
    if col in df.columns:
        print(f"\n--- Loss Ratio by {col} ---")
        g = eu.loss_ratio_by(df, col)
        print(g[['policies', 'loss_ratio', 'claim_frequency']].to_string())



--- Loss Ratio by Province ---
               policies  loss_ratio  claim_frequency
Province                                            
Western Cape        770    1.464109         0.200000
North West         1509    1.279878         0.188867
KwaZulu-Natal      4387    1.231101         0.182813
Limpopo            2363    1.219484         0.177740
Free State          278    1.193971         0.197842
Mpumalanga          574    0.999735         0.125436
Gauteng            2779    0.982672         0.155811
Northern Cape      2018    0.898879         0.123885
Eastern Cape       5322    0.885824         0.129651

--- Loss Ratio by VehicleType ---
                   policies  loss_ratio  claim_frequency
VehicleType                                             
Motorcycle              403    1.275813         0.166253
Passenger Vehicle     15625    1.078016         0.158400
Light Commercial       2966    1.056055         0.157788
Heavy Commercial       1006    1.026844         0.150099

--- Los

## 4. Univariate Distributions

In [7]:
fig = eu.plot_numeric_distributions(
    df,
    ['TotalPremium', 'TotalClaims', 'SumInsured', 'CalculatedPremiumPerTerm', 'CustomValueEstimate'],
    bins=60,
)
fig.suptitle("Key Financial Variable Distributions", y=1.01, fontsize=13, fontweight='bold')
fig.savefig(FIGURES / "premium_distribution.png", bbox_inches="tight", dpi=120)
plt.close()
print("Saved premium_distribution.png")


Saved premium_distribution.png


In [8]:
fig = eu.plot_categorical_counts(
    df,
    ['Province', 'Gender', 'CoverType', 'VehicleType', 'Make'],
    top_n=12,
)
fig.suptitle("Top Categories — Key Categorical Variables", y=1.01, fontsize=13, fontweight='bold')
fig.savefig(FIGURES / "categorical_counts.png", bbox_inches="tight", dpi=120)
plt.close()
print("Saved categorical_counts.png")


Saved categorical_counts.png


## 5. Outlier Detection

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['TotalClaims', 'TotalPremium', 'CustomValueEstimate']):
    sub = df[col].dropna()
    ax.boxplot(sub, vert=True, patch_artist=True,
               boxprops=dict(facecolor="#3a7ca5", alpha=0.5),
               medianprops=dict(color="black", linewidth=2),
               flierprops=dict(marker='o', markersize=2, alpha=0.3, color='#c44536'))
    ax.set_title(col)
    ax.set_ylabel("ZAR")
    pct99 = sub.quantile(0.995)
    ax.axhline(pct99, color='#c44536', linestyle='--', linewidth=1, label=f'p99.5 = R{pct99:,.0f}')
    ax.legend(fontsize=8)
fig.suptitle("Outlier Detection — Key Financial Variables", fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURES / "outliers_boxplot.png", bbox_inches="tight", dpi=120)
plt.close()
print("Saved outliers_boxplot.png")


Saved outliers_boxplot.png


## 6. Creative Visualisation 1 — Loss Ratio by Province

In [10]:
fig = eu.plot_loss_ratio_by(df, 'Province', top_n=9)
fig.savefig(FIGURES / "loss_ratio_by_province.png", bbox_inches="tight", dpi=120)
plt.close()
print("Saved loss_ratio_by_province.png")


Saved loss_ratio_by_province.png


## 7. Creative Visualisation 2 — Temporal Trends

In [11]:
fig, monthly = eu.plot_temporal_trends(df)
fig.savefig(FIGURES / "temporal_trends.png", bbox_inches="tight", dpi=120)
plt.close()
print("Saved temporal_trends.png")
monthly.head()


Saved temporal_trends.png


,policies,claims,severity,frequency
month,,,,
2014-02-01,1003,151,13080.867609,0.150548
2014-03-01,1100,153,13434.562022,0.139091
2014-04-01,1041,164,13756.366085,0.157541
2014-05-01,1077,163,13799.565758,0.151346
2014-06-01,1088,186,13067.540905,0.170956


## 8. Creative Visualisation 3 — Mean Claim by Vehicle Make

In [12]:
claim_by_make = (
    df[df['HasClaim'] == 1]
    .groupby('Make', observed=True)['TotalClaims']
    .agg(['mean', 'count'])
    .query('count >= 30')
    .sort_values('mean', ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=claim_by_make['mean'], y=claim_by_make.index.astype(str), ax=ax,
            palette="YlOrRd_r")
ax.set_xlabel("Mean Claim Amount (ZAR)")
ax.set_title("Mean Claim Severity by Vehicle Make (≥ 30 claims)", fontsize=12, fontweight='bold')
for i, v in enumerate(claim_by_make['mean']):
    ax.text(v + 50, i, f"R{v:,.0f}", va='center', fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES / "claim_by_make.png", bbox_inches="tight", dpi=120)
plt.close()
print("Saved claim_by_make.png")
claim_by_make


Saved claim_by_make.png


,mean,count
Make,,
Kia,14748.146205,352
Ford,14337.676710,329
Mazda,13732.234789,329
Toyota,13642.172372,377
Hyundai,13595.589026,333
Mercedes,13409.466568,362
Nissan,13355.584679,335
BMW,13306.039856,381
Volkswagen,12661.569222,363


## 9. Creative Visualisation 4 — Premium vs Claims by Postal Code

In [13]:
if 'PostalCode' in df.columns:
    by_zip = df.groupby('PostalCode', observed=True).agg(
        total_premium=('TotalPremium', 'sum'),
        total_claims=('TotalClaims', 'sum'),
        n_policies=('TotalPremium', 'size'),
    ).query('n_policies >= 50')

    fig, ax = plt.subplots(figsize=(9, 6))
    sc = ax.scatter(
        by_zip['total_premium'] / 1e6,
        by_zip['total_claims'] / 1e6,
        s=by_zip['n_policies'] / 3,
        c=by_zip['total_claims'] / by_zip['total_premium'].replace(0, np.nan),
        cmap='RdYlGn_r', alpha=0.75,
        vmin=0.5, vmax=1.5,
    )
    max_val = max(by_zip['total_premium'].max(), by_zip['total_claims'].max()) / 1e6
    ax.plot([0, max_val], [0, max_val], 'k--', linewidth=1.2, label='Break-even (LR=1)')
    plt.colorbar(sc, ax=ax, label='Loss Ratio')
    ax.set_xlabel("Total Premium (R millions)")
    ax.set_ylabel("Total Claims (R millions)")
    ax.set_title("Premium vs Claims by Postal Code\n(size ∝ exposure; colour = loss ratio)", fontsize=11, fontweight='bold')
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES / "premium_vs_claims_by_zip.png", bbox_inches="tight", dpi=120)
    plt.close()
    print("Saved premium_vs_claims_by_zip.png")
    by_zip.sort_values('total_claims', ascending=False).head(10)


Saved premium_vs_claims_by_zip.png


## 10. Summary of Key EDA Findings

| Finding | Value |
|---|---|
| Portfolio Loss Ratio | computed above |
| Claim Frequency | ~16% |
| Worst Province (loss ratio) | see chart |
| Best Province (loss ratio) | see chart |
| Top-risk vehicle make | see chart |
| Missing value rate | < 5% in any column |

**Data quality decisions:**
- `CustomValueEstimate` (~4% missing) → median imputed in modeling pipeline
- `Bank` (~2% missing) → mode imputed in pipeline
- Outliers in `TotalClaims` → winsorised at p99.5 in the cleaned dataset
- Rows with `TotalPremium = 0` → excluded from loss-ratio calculations
